In [30]:
# 1. IMPORTATION DES LIBRAIRIES
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from category_encoders import TargetEncoder
import optuna

In [31]:

# 1. CHARGEMENT ET CONCATÉNATION

# Chargement des datasets d'entraînement et de test
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

# Outliers à enlever selon la documentation
train_df = train_df[train_df["GrLivArea"] < 4000].reset_index(drop=True)
ntrain = len(train_df)
y_train_log = np.log1p(train_df["SalePrice"].copy())

all_data = pd.concat([train_df.drop(columns=["SalePrice"]), test_df], axis=0).reset_index(drop=True)

# Sauvegarde des IDs pour la soumission finale Kaggle, puis suppression
test_ids = test_df['Id'].copy()
all_data = all_data.drop(['Id'], axis=1)

print(f"Shape initiale all_data : {all_data.shape}")


# 2. NETTOYAGE GLOBAL SUR ALL_DATA

def apply_base_preprocessing(data):
    """Applique le nettoyage de base sur l'ensemble complet (Train + Test)"""
    df = data.copy()

    # 1. NA signifiant "Absence de l'équipement" -> Catégorie 'None'
    cols_none = ['Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1',
                 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish',
                 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType']
    for col in cols_none:
        if col in df.columns:
            df[col] = df[col].fillna('None')

    # 2. NA signifiant 0 pour les variables numériques
    cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2',
                 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
    for col in cols_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # 3. Imputation par la médiane globale pour la façade (LotFrontage)
    if 'LotFrontage' in df.columns:
        df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

    # 4. Imputation par le mode pour les vrais manquants
    cols_mode = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType', 'Functional', 'Utilities']
    for col in cols_mode:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    # 5. Traitement Multicolinéarité
    cols_to_drop = ['GarageArea', 'TotRmsAbvGrd', 'GarageYrBlt']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # 6. Feature Engineering (Agressif et optimisé pour les Arbres)
    
    # Surfaces combinées
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalPorchSF'] = df.get('OpenPorchSF', 0) + df.get('EnclosedPorch', 0) + df.get('3SsnPorch', 0) + df.get('ScreenPorch', 0)
    
    # Équipements combinés (Salles de bain)
    df['TotalBaths'] = df.get('FullBath', 0) + 0.5 * df.get('HalfBath', 0) + df.get('BsmtFullBath', 0) + 0.5 * df.get('BsmtHalfBath', 0)
    
    # Interactions puissantes (Les arbres adorent ça car ils ne savent pas multiplier eux-mêmes)
    # OverallQual et OverallCond sont des entiers (1-10) dans le dataset brut
    if 'OverallQual' in df.columns:
        df['Qual_x_Size'] = df['OverallQual'] * df['TotalSF']
        df['OverallScore'] = df['OverallQual'] * df.get('OverallCond', 1)
        
    # Variables de temps et Rénovation
    df['AgeAtSale'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    
    # Suppression des années d'origine (anti-colinéarité)
    cols_to_drop_years = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'YrSold']
    df = df.drop(columns=[c for c in cols_to_drop_years if c in df.columns])

    return df

# Application du nettoyage
all_data_clean = apply_base_preprocessing(all_data)


# 3. SÉPARATION TRAIN / TEST

# On utilise ntrain pour retrouver nos données d'entraînement exactes
X_train_clean = all_data_clean.iloc[:ntrain].copy()
X_test_clean = all_data_clean.iloc[ntrain:].copy()

print(f"Shape X_train_clean : {X_train_clean.shape}")
print(f"Shape X_test_clean  : {X_test_clean.shape}")

Shape initiale all_data : (2915, 79)
Shape X_train_clean : (1456, 81)
Shape X_test_clean  : (1459, 81)


In [32]:

# 4. ENCODAGE AVANCÉ

class AdvancedCategoricalEngineer(BaseEstimator, TransformerMixin):
    """Transformateur Scikit-Learn pour encodage ordinal et Target Encoding"""
    def __init__(self, smoothing=10.0):
        self.smoothing = smoothing
        self.qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
        self.ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                             'HeatingQC', 'KitchenQual', 'FireplaceQu',
                             'GarageQual', 'GarageCond', 'PoolQC']
        self.te = None
        self.nominal_cols = None

    def fit(self, X, y):
        X_copy = X.copy()
        for col in self.ordinal_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.qual_mapping).fillna(0)

        self.nominal_cols = X_copy.select_dtypes(include=['object']).columns.tolist()

        for col in self.nominal_cols:
            X_copy[col] = X_copy[col].astype(str).fillna('Missing')

        self.te = TargetEncoder(cols=self.nominal_cols, smoothing=self.smoothing)
        self.te.fit(X_copy[self.nominal_cols], y)
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.ordinal_cols:
            if col in X_copy.columns:
                X_copy[col] = X_copy[col].map(self.qual_mapping).fillna(0)

        if self.nominal_cols:
            for col in self.nominal_cols:
                if col in X_copy.columns:
                    X_copy[col] = X_copy[col].astype(str).fillna('Missing')
            X_copy[self.nominal_cols] = self.te.transform(X_copy[self.nominal_cols])

        return X_copy


# 5. VALIDATION CROISÉE

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

print("\nDébut Validation Croisée (5-Fold)...")
# Création du pipeline M1
pipeline_m1 = Pipeline([
    ('encoder', AdvancedCategoricalEngineer(smoothing=10.0)),
    ('model', XGBRegressor(n_estimators=600, learning_rate=0.04, max_depth=5,
                           reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1))
])

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_clean)):

    # Séparation locale pour ce pli
    X_tr, y_tr = X_train_clean.iloc[train_idx], y_train_log.iloc[train_idx]
    X_val, y_val = X_train_clean.iloc[val_idx], y_train_log.iloc[val_idx]

    # Entraînement et Prédiction
    pipeline_m1.fit(X_tr, y_tr)
    preds = pipeline_m1.predict(X_val)

    fold_rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_scores.append(fold_rmse)
    print(f"Fold {fold + 1} | RMSE: {fold_rmse:.5f}")

print(f"\nRMSE Moyen (CV) : {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})")


Début Validation Croisée (5-Fold)...
Fold 1 | RMSE: 0.14004
Fold 2 | RMSE: 0.11797
Fold 3 | RMSE: 0.13462
Fold 4 | RMSE: 0.13335
Fold 5 | RMSE: 0.10417

RMSE Moyen (CV) : 0.12603 (+/- 0.01316)


In [33]:
# Désactiver les logs d'Optuna pour ne pas polluer l'écran
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """Fonction d'objectif pour Optuna : trouve les meilleurs paramètres XGBoost avec Early Stopping"""

    # 1. Grille de paramètres à tester par Optuna
    param = {
        'n_estimators': 3000, # Fixé haut pour l'early stopping
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist', # Optimisation par histogramme
        'early_stopping_rounds': 100 # L'arbre s'arrête si pas d'amélioration pdt 100 tours
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    # 2. Boucle manuelle (indispensable pour l'early stopping)
    for train_idx, val_idx in kf.split(X_train_clean):
        X_tr, y_tr = X_train_clean.iloc[train_idx], y_train_log.iloc[train_idx]
        X_val, y_val = X_train_clean.iloc[val_idx], y_train_log.iloc[val_idx]
        
        # Encodage manuel pour éviter le data leakage et permettre le eval_set
        encoder = AdvancedCategoricalEngineer(smoothing=10.0)
        X_tr_enc = encoder.fit_transform(X_tr, y_tr)
        X_val_enc = encoder.transform(X_val)
        
        model = XGBRegressor(**param)
        
        # Entraînement avec validation set pour early stopping
        model.fit(
            X_tr_enc, y_tr,
            eval_set=[(X_val_enc, y_val)],
            verbose=False
        )
        
        # On sauvegarde le meilleur score atteint à la meilleure itération
        cv_scores.append(model.best_score)

    return np.mean(cv_scores)

# LANCEMENT DE L'OPTIMISATION (50 essais)
print("Lancement d'Optuna : Recherche des meilleurs paramètres avec Early Stopping (patienter un peu...)")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("\n--- RÉSULTATS OPTIMISATION ---")
print(f"Meilleur RMSE (CV) : {study.best_value:.5f}")
print("Meilleurs Paramètres trouvés :")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Lancement d'Optuna : Recherche des meilleurs paramètres avec Early Stopping (patienter un peu...)

--- RÉSULTATS OPTIMISATION ---
Meilleur RMSE (CV) : 0.11652
Meilleurs Paramètres trouvés :
  learning_rate: 0.011241717744465685
  max_depth: 3
  min_child_weight: 5
  subsample: 0.6548008398568045
  colsample_bytree: 0.7925874843462873
  reg_alpha: 0.1111988452019818
  reg_lambda: 3.087524732188881


In [34]:
# ÉTAPE FINALE : ENTRAÎNEMENT DU MODÈLE FINAL ET SOUMISSION
from sklearn.model_selection import train_test_split

print("1 & 2. Création et entraînement du Modèle Final avec les paramètres Optuna...")

# On récupère AUTOMATIQUEMENT les meilleurs paramètres trouvés par Optuna
final_params = study.best_params.copy()

# On rajoute les paramètres fixes
final_params['random_state'] = 42
final_params['n_jobs'] = -1
final_params['tree_method'] = 'hist'
final_params['n_estimators'] = 3000
final_params['early_stopping_rounds'] = 100

# Pour l'early stopping final, on garde 5% du train
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_train_clean, y_train_log, test_size=0.05, random_state=42
)

# On encode explicitement pour éviter d'utiliser Pipeline (incompatible avec eval_set simple)
final_encoder = AdvancedCategoricalEngineer(smoothing=10.0)
X_train_final_enc = final_encoder.fit_transform(X_train_final, y_train_final)
X_val_final_enc = final_encoder.transform(X_val_final)
X_test_clean_enc = final_encoder.transform(X_test_clean)

final_model = XGBRegressor(**final_params)

# Entraînement sur 95% du Train avec Early Stopping
final_model.fit(
    X_train_final_enc, y_train_final,
    eval_set=[(X_val_final_enc, y_val_final)],
    verbose=False
)

print(f"Modèle final arrêté optimalement à l'arbre n°{final_model.best_iteration}")

print("3. Prédictions sur le jeu de test Kaggle...")
log_predictions = final_model.predict(X_test_clean_enc)

# Transformation inverse (Expm1) pour revenir en Dollars ($)
final_predictions = np.expm1(log_predictions)

print("4. Création du fichier de soumission...")
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

# Sauvegarde au format CSV
submission.to_csv('submission_M1_4.2_XGBoost.csv', index=False)

print("\nC'est terminé ! Le fichier 'submission_M1_4.2_XGBoost.csv' est prêt à être soumis sur Kaggle.")
display(submission.head())

1 & 2. Création et entraînement du Modèle Final avec les paramètres Optuna...
Modèle final arrêté optimalement à l'arbre n°1409
3. Prédictions sur le jeu de test Kaggle...
4. Création du fichier de soumission...

C'est terminé ! Le fichier 'submission_M1_4.2_XGBoost.csv' est prêt à être soumis sur Kaggle.


,Id,SalePrice
0,1461,122980.304688
1,1462,156160.468750
2,1463,180680.359375
3,1464,191300.656250
4,1465,189611.640625
